# 🎵 Estudio de Música Digital — Separar instrumentos con IA

Este notebook separa una canción en pistas individuales (voz, batería, bajo, piano, guitarra, otros) usando **Demucs** (Meta AI), el mejor modelo abierto para esto hoy.

**Por qué en Colab y no en el teléfono/servidor de VAWOL:** este modelo necesita GPU para andar en minutos en vez de correr el riesgo de colgarse por falta de RAM. Colab te da una GPU gratis (sin tarjeta de crédito) — este es el camino de costo cero.

Hay **dos formas** de usarlo (elegí una sección más abajo):

- **A) Manual**: subís una canción, esperás, se descarga un .zip. Simple, un solo uso.
- **B) Servidor vivo**: el notebook queda corriendo y expone una URL pública — desde ahí, **arte.vawol.com** (o el agente Jano) le puede pedir separar canciones directamente, sin bajar/subir nada a mano, mientras esta pestaña de Colab siga abierta.

**Antes que nada**: `Entorno de ejecución` → `Cambiar tipo de entorno de ejecución` → elegí **GPU (T4)**.

In [ ]:
# Verificar que hay GPU (si dice "no GPU", volvé al paso de arriba)
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader || echo "⚠️ SIN GPU: anda a Entorno de ejecución > Cambiar tipo > GPU"

In [ ]:
# Instalar Demucs (tarda ~1-2 minutos)
!pip install -q demucs

## A) Modo manual — subir una canción y bajar el .zip

In [ ]:
# Subir la canción
from google.colab import files
import os

subidos = files.upload()
nombre_archivo = list(subidos.keys())[0]
print(f"\n✓ Subido: {nombre_archivo}")

### Elegí el modelo

- **`htdemucs_6s`** (recomendado si querés el **piano** o la **guitarra** separados): 6 pistas — voz, batería, bajo, guitarra, piano, otros. Es un modelo experimental; a veces se "filtra" un poco de otro instrumento en la pista de piano/guitarra.
- **`htdemucs_ft`**: el de mejor calidad general, pero solo 4 pistas — voz, batería, bajo, otros (el piano queda mezclado dentro de "otros"). Tarda un poco más (corre 4 modelos y promedia).

In [ ]:
MODELO = "htdemucs_6s"  # cambiar a "htdemucs_ft" si no necesitás piano/guitarra separados

!demucs -n {MODELO} "{nombre_archivo}"

In [ ]:
# Empaquetar los resultados y descargar
import shutil

base = os.path.splitext(nombre_archivo)[0]
carpeta = f"separated/{MODELO}/{base}"
print("Pistas generadas:", os.listdir(carpeta))

zip_path = f"{base}_instrumentos"
shutil.make_archive(zip_path, "zip", carpeta)
files.download(f"{zip_path}.zip")

---
## B) Modo servidor vivo — para usar directo desde arte.vawol.com

Levanta un mini-servidor (FastAPI) con un endpoint `/separar`, y lo expone a
internet con **cloudflared** (el mismo túnel que ya usamos en el nodo del
teléfono). Mientras esta celda siga corriendo, la URL pública que imprime
puede recibir pedidos de separación reales — el sitio o el MCP server se la
pasan y les devuelve el .zip de instrumentos, sin que nadie tenga que bajar
ni subir nada a mano.

**Importante**: esto NO es un servicio 24/7 — solo funciona mientras tengas
esta pestaña de Colab abierta y conectada. Si se corta la sesión, hay que
volver a correr esta celda (te da una URL nueva cada vez).

In [ ]:
%%writefile servidor.py
# API de separacion: los trabajos corren en segundo plano y se consultan por
# id — asi ninguna conexion queda abierta minutos (los proxies de internet
# cortan a los ~100s y el modelo htdemucs_ft tarda varios minutos).
# POST /separar-async -> {"trabajo": id}   GET /estado/{id}   GET /resultado/{id}
# (POST /separar sincrono queda por compatibilidad para audios cortos.)
from fastapi import FastAPI, UploadFile, File, Form
from fastapi.middleware.cors import CORSMiddleware
from fastapi.responses import FileResponse, JSONResponse
import subprocess, os, shutil, uuid, threading

app = FastAPI()
app.add_middleware(CORSMiddleware, allow_origins=["*"], allow_methods=["*"], allow_headers=["*"])

TRABAJOS = {}  # id -> {"estado": "procesando"|"listo"|"error", "zip": ruta, "detalle": str}

@app.get("/salud")
def salud():
    # "version" permite confirmar desde afuera que esta corriendo el codigo
    # mas nuevo (antes "async: true" no alcanzaba para distinguir versiones).
    return {"ok": True, "async": True, "version": "mp3-stems-v1"}

def _preparar(archivo):
    trabajo = str(uuid.uuid4())
    carpeta = f"/tmp/{trabajo}"
    os.makedirs(carpeta, exist_ok=True)
    nombre = os.path.basename(archivo.filename or "cancion.mp3")
    ruta = os.path.join(carpeta, nombre)
    with open(ruta, "wb") as f:
        shutil.copyfileobj(archivo.file, f)
    return trabajo, carpeta, ruta

def _separar(trabajo, carpeta, ruta, modelo):
    try:
        # Stems en MP3: 6 WAVs de una cancion de 4 min pesan ~250MB y por el tunel
        # tardan una eternidad (en un telefono directamente muere). En MP3 192k
        # el zip queda ~30MB con calidad excelente.
        subprocess.run(["demucs", "--mp3", "--mp3-bitrate", "192", "-n", modelo, "-o", carpeta, ruta], check=True, capture_output=True)
        base = os.path.splitext(os.path.basename(ruta))[0]
        salida = os.path.join(carpeta, modelo, base)
        zip_path = shutil.make_archive(os.path.join(carpeta, "instrumentos"), "zip", salida)
        TRABAJOS[trabajo] = {"estado": "listo", "zip": zip_path}
    except Exception as e:
        TRABAJOS[trabajo] = {"estado": "error", "detalle": str(e)[:300]}

@app.post("/separar-async")
async def separar_async(archivo: UploadFile = File(...), modelo: str = Form("htdemucs_6s")):
    trabajo, carpeta, ruta = _preparar(archivo)
    TRABAJOS[trabajo] = {"estado": "procesando"}
    threading.Thread(target=_separar, args=(trabajo, carpeta, ruta, modelo), daemon=True).start()
    return {"trabajo": trabajo}

@app.get("/estado/{trabajo}")
def estado(trabajo: str):
    t = TRABAJOS.get(trabajo)
    if not t:
        return JSONResponse({"estado": "desconocido"}, status_code=404)
    return {"estado": t["estado"], "detalle": t.get("detalle", "")}

@app.get("/resultado/{trabajo}")
def resultado(trabajo: str):
    t = TRABAJOS.get(trabajo)
    if not t or t["estado"] != "listo":
        return JSONResponse({"error": "El trabajo no está listo."}, status_code=404)
    return FileResponse(t["zip"], media_type="application/zip", filename="instrumentos.zip")

@app.post("/separar")
async def separar(archivo: UploadFile = File(...), modelo: str = Form("htdemucs_6s")):
    trabajo, carpeta, ruta = _preparar(archivo)
    _separar(trabajo, carpeta, ruta, modelo)
    t = TRABAJOS[trabajo]
    if t["estado"] != "listo":
        return JSONResponse({"error": t.get("detalle", "fallo la separacion")}, status_code=500)
    return FileResponse(t["zip"], media_type="application/zip", filename="instrumentos.zip")


In [ ]:
# Instalar dependencias del servidor + cloudflared, y levantar todo
!pip install -q fastapi uvicorn python-multipart lameenc
!test -f cloudflared || (wget -q -O cloudflared https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 && chmod +x cloudflared)

import subprocess, time, re

# Si quedo un servidor/tunel de una corrida anterior, lo bajamos: si no, el
# puerto 8000 queda ocupado y el tunel nuevo apuntaria al servidor VIEJO.
subprocess.run(["pkill", "-f", "uvicorn"], capture_output=True)
subprocess.run(["pkill", "-f", "cloudflared"], capture_output=True)
time.sleep(2)

proceso_servidor = subprocess.Popen(
    ["uvicorn", "servidor:app", "--host", "0.0.0.0", "--port", "8000"],
    stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL,
)
time.sleep(4)

proceso_tunel = subprocess.Popen(
    ["./cloudflared", "tunnel", "--url", "http://localhost:8000"],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
)

url_publica = None
for _ in range(40):
    linea = proceso_tunel.stdout.readline()
    m = re.search(r"https://[a-zA-Z0-9\-]+\.trycloudflare\.com", linea)
    if m:
        url_publica = m.group(0)
        break

if url_publica:
    print("✓ Servidor de separación activo en:", url_publica)
    # Auto-registro: si guardaste el secreto en Colab (🔑 Secrets, nombre
    # VAWOL_GPU_TOKEN), le avisamos a sura y la "GPU de VAWOL" queda viva en
    # arte.vawol.com sin pegar nada a mano. Sin el secreto, se pega manual.
    try:
        from google.colab import userdata
        token = userdata.get("VAWOL_GPU_TOKEN").strip()
        import urllib.request, json as _json
        pedido = urllib.request.Request(
            "https://app.vawol.com/hcgi/api/estudio/gpu-url",
            data=_json.dumps({"url": url_publica}).encode(),
            headers={"Content-Type": "application/json", "X-Gpu-Token": token},
            method="POST",
        )
        with urllib.request.urlopen(pedido, timeout=15) as r:
            _json.loads(r.read())
        print("✓ Registrada como GPU de VAWOL — arte.vawol.com ya la usa solo.")
    except Exception as e:
        print("  (Sin auto-registro:", str(e)[:80], ")")
        print("  Pegá la URL en arte.vawol.com (Separar con IA → Colab manual) o agregá el secreto VAWOL_GPU_TOKEN en Colab.")
else:
    print("⚠️ No pude leer la URL del túnel — revisá la salida de arriba.")

In [ ]:
# Correr esta celda para apagar el servidor y el túnel cuando termines
proceso_tunel.terminate()
proceso_servidor.terminate()
print("Servidor y túnel detenidos.")

---
### Notas honestas
- **Tiempo**: con GPU T4, una canción de 3-4 minutos tarda típicamente 1-3 minutos en separarse (vs. varios minutos por CPU, sin GPU).
- **Cuota gratis de Colab**: variable según uso de Google, pero suele rondar 12-30 horas de GPU por semana sin pagar nada. Para separar canciones sueltas alcanza de sobra.
- **Calidad**: Demucs es el estado del arte abierto hoy, pero ningún separador es perfecto — vas a escuchar algo de "bleed" (residuos de otros instrumentos), sobre todo en el modelo de 6 pistas.
- **El modo servidor no es 24/7**: necesita que tengas la pestaña de Colab abierta y conectada. Si se corta la sesión (por inactividad o límite de tiempo), hay que volver a correr las celdas — te va a dar una URL nueva cada vez.